# 01 — Data Preparation (Canonical Pipeline)

**Purpose.** This is the single source of truth for train/val/test data used by every fine-tuning
notebook in this project (mT5, AraBART, AraT5, Jais-QLoRA). No other notebook should re-derive
its own filtering — they all load the files this notebook produces.

**Why this notebook exists.** An earlier iteration of this project had preprocessing logic
duplicated across notebooks, which drifted out of sync with what was actually used for training
(documented filter thresholds didn't match the row counts actually loaded during training).
This notebook fixes that by being the *only* place filtering happens, and by writing out a
`manifest.json` that records exactly what was done, so results are always traceable back to a
specific run of this notebook.

**Key design decisions (see inline markdown for rationale at each step):**
1. Combine AraSum (DW Arabic) + XL-Sum (BBC Arabic) into one pool.
2. Deduplicate *within* each source and *across* the combined pool.
3. Apply a length-ratio filter to remove degenerate pairs (junk captions, near-empty summaries).
4. Apply a **symmetric** lexical-overlap filter — drop near-verbatim copies AND near-unrelated
   pairs, but *keep* the full spectrum in between (including genuinely paraphrastic,
   low-overlap-but-good summaries). This is a deliberate change from filtering toward
   *high*-overlap-only pairs, which would bias the corpus toward extractive-style summaries and
   undercut any claim about ROUGE undervaluing abstraction.
5. Carve out **fixed, hash-verified, non-overlapping** train/val/test splits, sized to be
   statistically comparable to the original AraSum paper's evaluation (Kahla et al., 2023, used
   a ~3,969-example test set) rather than a 100-example spot-check.


## Step 0 — Setup

In [ ]:
!pip install -q rapidfuzz


In [ ]:
import pandas as pd
import numpy as np
import hashlib
import json
import platform
import os
import tarfile
import urllib.request
from datetime import datetime, timezone
import matplotlib.pyplot as plt
from rapidfuzz import fuzz
from tqdm import tqdm

tqdm.pandas()

RANDOM_SEED = 42
N_TEST = 1000
N_VAL = 500

# Quality-filter thresholds — see markdown rationale below for each
LENGTH_RATIO_MIN = 2       # article_len / summary_len below this = summary too close to article length
LENGTH_RATIO_MAX = 20      # above this = summary likely a junk caption, not a real summary
SIMILARITY_MIN = 10        # at/below this = article and summary likely misaligned or unrelated
SIMILARITY_MAX = 95        # at/above this = summary is essentially a verbatim copy of part of the article

OUTPUT_DIR = "/kaggle/working/canonical_dataset"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Config: seed={RANDOM_SEED}, test={N_TEST}, val={N_VAL}")
print(f"Length ratio filter: [{LENGTH_RATIO_MIN}, {LENGTH_RATIO_MAX}]")
print(f"Similarity filter (exclusive): ({SIMILARITY_MIN}, {SIMILARITY_MAX})")


## Step 1 — Load AraSum

Update `ARASUM_PATH` to wherever the raw AraSum TSV is attached in your Kaggle input.

In [ ]:
ARASUM_PATH = "/kaggle/input/arasum/AbsArSumCorpus_DW_article tab lead.csv"

arasum_raw = pd.read_csv(ARASUM_PATH, delimiter="\t", encoding="utf-8")
arasum_raw.columns = ["id", "article", "summary"]
arasum_raw = arasum_raw.drop(columns=["id"])
arasum_raw["source"] = "arasum"

print(f"AraSum raw: {len(arasum_raw)} rows")
arasum_raw.head(2)


## Step 2 — Load XL-Sum (Arabic)

**Not using `datasets.load_dataset()` here.** As of `datasets>=4.0`, script-based dataset
loading was removed entirely (no `trust_remote_code` workaround), and `csebuetnlp/xlsum` still
ships a `.py` loading script rather than a Parquet export — so `load_dataset("csebuetnlp/xlsum",
"arabic")` now fails with `RuntimeError: Dataset scripts are no longer supported`, regardless of
which flags you pass. This isn't version-pinning-fixable in a durable way (Kaggle's preinstalled
`datasets` version changes over time and this dataset may never get converted to Parquet).

Instead, this downloads the exact same archive `xlsum.py` itself downloads — confirmed straight
from the script source
(`https://huggingface.co/datasets/csebuetnlp/xlsum/blob/main/xlsum.py`) — and parses the JSONL
files directly. No dependency on `datasets` library internals at all, so it can't break the same
way again.

This also pulls in XL-Sum's **train + validation + test** splits combined (not just train, as
before) — since this notebook builds its own principled train/val/test split downstream anyway,
there's no reason to discard ~20% of the available Arabic data.

In [ ]:
XLSUM_URL = "https://huggingface.co/datasets/csebuetnlp/xlsum/resolve/main/data/arabic_XLSum_v2.0.tar.bz2"
XLSUM_CACHE_DIR = "/kaggle/working/xlsum_arabic"
os.makedirs(XLSUM_CACHE_DIR, exist_ok=True)
archive_path = f"{XLSUM_CACHE_DIR}/arabic_XLSum_v2.0.tar.bz2"

if not os.path.exists(archive_path):
    print(f"Downloading {XLSUM_URL} ...")
    urllib.request.urlretrieve(XLSUM_URL, archive_path)
    print("Download complete.")
else:
    print("Archive already downloaded, skipping.")

with tarfile.open(archive_path, "r:bz2") as tar:
    tar.extractall(XLSUM_CACHE_DIR, filter="data")
    extracted_files = tar.getnames()
print("Extracted:", extracted_files)


In [ ]:
xlsum_parts = []
for split_suffix in ["train", "val", "test"]:
    jsonl_path = f"{XLSUM_CACHE_DIR}/arabic_{split_suffix}.jsonl"
    part = pd.read_json(jsonl_path, lines=True)
    xlsum_parts.append(part)
    print(f"  arabic_{split_suffix}.jsonl: {len(part)} rows")

xlsum_raw = pd.concat(xlsum_parts, ignore_index=True)[["text", "summary"]].rename(columns={"text": "article"})
xlsum_raw["source"] = "xlsum"

print(f"\nXL-Sum (Arabic, train+val+test combined) raw: {len(xlsum_raw)} rows")
xlsum_raw.head(2)


## Step 3 — Clean each source independently

Remove nulls, exact-duplicate (article, summary) pairs, and rows where article and summary are
identical (these provide no summarization signal and shouldn't count as either train or eval
examples).

In [ ]:
def clean_source(df, name):
    before = len(df)
    df = df.dropna(subset=["article", "summary"]).copy()
    df["article"] = df["article"].astype(str).str.strip()
    df["summary"] = df["summary"].astype(str).str.strip()
    df = df[(df["article"].str.len() > 0) & (df["summary"].str.len() > 0)]
    df = df.drop_duplicates(subset=["article", "summary"], keep="first")
    df = df[df["article"] != df["summary"]]
    print(f"[{name}] {before} -> {len(df)} rows after null/dup/self-match cleaning")
    return df.reset_index(drop=True)

arasum_clean = clean_source(arasum_raw, "AraSum")
xlsum_clean = clean_source(xlsum_raw, "XL-Sum")


## Step 4 — Combine and deduplicate across sources

AraSum is sourced from Deutsche Welle Arabic and XL-Sum from BBC Arabic. Different outlets can
still cover the same wire-service story (AP/Reuters/AFP), so we deduplicate the *combined* pool
by exact article text, not just within each source.

In [ ]:
combined = pd.concat([arasum_clean, xlsum_clean], ignore_index=True)
print(f"Combined (pre cross-source dedup): {len(combined)} rows")

combined["article_hash"] = combined["article"].apply(
    lambda t: hashlib.sha256(t.encode("utf-8")).hexdigest()
)
before = len(combined)
combined = combined.drop_duplicates(subset=["article_hash"], keep="first").reset_index(drop=True)
print(f"Combined (post cross-source dedup): {before} -> {len(combined)} rows")


## Step 5 — Length-ratio filter

Removes two degenerate cases: summaries nearly as long as their article (uninformative
compression), and summaries that are tiny fragments/captions relative to the article (often
non-summary content like video captions, e.g. `"*هذا الفيديو بدون تعليق*"`).

In [ ]:
combined["length_ratio"] = (
    combined["article"].str.len() / combined["summary"].str.len().replace(0, np.nan)
)
combined = combined.dropna(subset=["length_ratio"])

plt.figure(figsize=(12, 4))
plt.hist(combined["length_ratio"].clip(upper=40), bins=100, color="steelblue", edgecolor="black", alpha=0.7)
plt.axvline(LENGTH_RATIO_MIN, color="green", linestyle="--", label=f"min={LENGTH_RATIO_MIN}")
plt.axvline(LENGTH_RATIO_MAX, color="red", linestyle="--", label=f"max={LENGTH_RATIO_MAX}")
plt.xlabel("Article-to-summary length ratio (clipped at 40 for display)")
plt.ylabel("Count")
plt.title("Length ratio distribution")
plt.legend()
plt.show()

before = len(combined)
combined = combined[
    (combined["length_ratio"] >= LENGTH_RATIO_MIN) & (combined["length_ratio"] <= LENGTH_RATIO_MAX)
].copy()
print(f"After length-ratio filter: {before} -> {len(combined)} rows")


## Step 6 — Symmetric lexical-overlap filter

**This is the key methodological fix versus the earlier pipeline.** We compute
`token_set_ratio(article, summary)` as a lexical-overlap score, exactly as before — but instead
of *keeping only high-overlap pairs* (which biases the corpus toward extractive, near-copy
summaries and removes the paraphrastic cases that motivate this whole project), we drop only the
two degenerate tails:

- **similarity ≥ 95**: the summary is essentially a verbatim excerpt of the article — not a
  useful abstractive-summarization training/eval example.
- **similarity ≤ 10**: article and summary share almost no vocabulary — likely a misalignment
  or data error, not a genuine (if heavily paraphrased) summary.

Everything in between is kept, preserving the full extractive-to-abstractive spectrum this
project needs to say anything meaningful about ROUGE's behavior on Arabic paraphrase.

In [ ]:
combined["similarity"] = combined.progress_apply(
    lambda row: fuzz.token_set_ratio(row["article"], row["summary"]), axis=1
)

plt.figure(figsize=(12, 4))
plt.hist(combined["similarity"], bins=100, color="darkorange", edgecolor="black", alpha=0.7)
plt.axvline(SIMILARITY_MIN, color="green", linestyle="--", label=f"min={SIMILARITY_MIN}")
plt.axvline(SIMILARITY_MAX, color="red", linestyle="--", label=f"max={SIMILARITY_MAX}")
plt.xlabel("Article-summary lexical overlap (token_set_ratio)")
plt.ylabel("Count")
plt.title("Similarity distribution — symmetric filter keeps the middle, drops both tails")
plt.legend()
plt.show()

before = len(combined)
combined = combined[
    (combined["similarity"] > SIMILARITY_MIN) & (combined["similarity"] < SIMILARITY_MAX)
].copy()
print(f"After symmetric similarity filter: {before} -> {len(combined)} rows")
print(combined["similarity"].describe())


## Step 7 — Fixed splits with leakage verification

Test and validation sets are carved out **once, here**, and never touched again. Every
fine-tuning notebook loads these exact files — nothing is re-sampled or re-filtered downstream,
which is what caused the train/test overlap risk in the earlier pipeline.

Test set size (1,000) is far larger than a 100-example spot-check and is a subsample of the same
pool the original AraSum paper (Kahla et al., 2023) drew its own ~3,969-example test set from —
close enough in spirit to be a fair comparison point, cheap enough to evaluate in minutes.

In [ ]:
shuffled = combined.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

test_df = shuffled.iloc[:N_TEST].copy()
val_df = shuffled.iloc[N_TEST:N_TEST + N_VAL].copy()
train_df = shuffled.iloc[N_TEST + N_VAL:].copy()

# Hash-based leakage check — this is the assertion that was missing before
train_hashes = set(train_df["article_hash"])
val_hashes = set(val_df["article_hash"])
test_hashes = set(test_df["article_hash"])

assert train_hashes.isdisjoint(val_hashes), "LEAK DETECTED: train/val article overlap"
assert train_hashes.isdisjoint(test_hashes), "LEAK DETECTED: train/test article overlap"
assert val_hashes.isdisjoint(test_hashes), "LEAK DETECTED: val/test article overlap"

print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
print("No article-level overlap between splits — verified by hash comparison.")

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"  {name}: source breakdown -> {d['source'].value_counts().to_dict()}")


## Step 8 — Save canonical files + manifest

In [ ]:
cols_to_save = ["article", "summary", "source", "similarity", "length_ratio", "article_hash"]

train_df[cols_to_save].to_csv(f"{OUTPUT_DIR}/train.csv", index=False)
val_df[cols_to_save].to_csv(f"{OUTPUT_DIR}/val.csv", index=False)
test_df[cols_to_save].to_csv(f"{OUTPUT_DIR}/test.csv", index=False)

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "random_seed": RANDOM_SEED,
    "sources": {
        "arasum_raw_rows": int(len(arasum_raw)),
        "arasum_after_cleaning": int(len(arasum_clean)),
        "xlsum_raw_rows": int(len(xlsum_raw)),
        "xlsum_after_cleaning": int(len(xlsum_clean)),
    },
    "filters": {
        "length_ratio_min": LENGTH_RATIO_MIN,
        "length_ratio_max": LENGTH_RATIO_MAX,
        "similarity_min_exclusive": SIMILARITY_MIN,
        "similarity_max_exclusive": SIMILARITY_MAX,
        "filter_type": "symmetric (drops both near-verbatim-copy and near-unrelated pairs)",
    },
    "final_pool_size": int(len(combined)),
    "splits": {
        "train": int(len(train_df)),
        "val": int(len(val_df)),
        "test": int(len(test_df)),
    },
    "leakage_check": "passed (hash-based, asserted at build time)",
    "notes": (
        "This manifest is the authoritative record of how train/val/test were built. "
        "Every fine-tuning notebook should read train.csv/val.csv/test.csv from the Kaggle "
        "dataset built from this OUTPUT_DIR and must not re-filter or re-sample them."
    ),
}

with open(f"{OUTPUT_DIR}/manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print(json.dumps(manifest, ensure_ascii=False, indent=2))


## Next step

Upload the contents of `/kaggle/working/canonical_dataset/` (`train.csv`, `val.csv`, `test.csv`,
`manifest.json`) as a **new, versioned Kaggle dataset** (e.g. `arabic-summ-canonical-v1`).

Every notebook in `02_finetune_*.ipynb` loads from that dataset by version — if you ever change
a filter threshold here, bump the dataset version rather than overwriting it, so old results stay
reproducible against the data that actually produced them.